<a href="https://colab.research.google.com/github/Ashu251023/DETraining/blob/Dev/SPARKUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install Java, Spark, and required Python packages
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.4.1/spark-3.4.1-bin-hadoop3.tgz
!tar xf spark-3.4.1-bin-hadoop3.tgz
!pip install -q findspark pyngrok


In [2]:
import os
import findspark
from pyngrok import ngrok, conf

# Set environment variables
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.4.1-bin-hadoop3"

# Initialize Spark
findspark.init()


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ColabSparkUI") \
    .config("spark.ui.port", "4040") \
    .getOrCreate()


In [4]:
df = spark.range(1000)
df.count()  # Required to start Spark UI


1000

In [5]:
# Replace with your actual ngrok authtoken
conf.get_default().auth_token = "2yAjXOo7HA7ks5nkLbJh8SwgdAL_5DusQoCAaWQN9koq2yfuW"

# Kill old tunnels (if any) and start new one
ngrok.kill()
public_url = ngrok.connect(4040)
print(f"✅ Spark UI is available at: {public_url}")


✅ Spark UI is available at: NgrokTunnel: "https://5a341b3bad8e.ngrok-free.app" -> "http://localhost:4040"


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MultiStageExample").getOrCreate()
sc = spark.sparkContext

# Create an RDD
rdd = sc.parallelize(range(1, 10001), 4)

# Stage 1: map (narrow transformation, no shuffle)
mapped_rdd = rdd.map(lambda x: (x % 10, x))

# Stage 2: reduceByKey (wide transformation, triggers shuffle)
reduced_rdd = mapped_rdd.reduceByKey(lambda a, b: a + b)

# Stage 3: map again (narrow transformation, no shuffle)
final_rdd = reduced_rdd.map(lambda x: (x[0], x[1] * 2))

# Action to trigger execution
result = final_rdd.collect()

print(result[:10])  # Show some output


[(4, 9998000), (8, 10006000), (0, 10010000), (1, 9992000), (5, 10000000), (9, 10008000), (2, 9994000), (6, 10002000), (3, 9996000), (7, 10004000)]


In [ ]:
df = spark.range(10000)

# Stage 1: filter (narrow)
filtered_df = df.filter("id % 2 = 0")

# Stage 2: groupBy + agg (wide shuffle)
grouped_df = filtered_df.groupBy("id").count()

# Stage 3: select (narrow)
final_df = grouped_df.select("id", "count")

final_df.show()


+---+-----+
| id|count|
+---+-----+
|  0|    1|
|  2|    1|
|  4|    1|
|  6|    1|
|  8|    1|
| 10|    1|
| 12|    1|
| 14|    1|
| 16|    1|
| 18|    1|
| 20|    1|
| 22|    1|
| 24|    1|
| 26|    1|
| 28|    1|
| 30|    1|
| 32|    1|
| 34|    1|
| 36|    1|
| 38|    1|
+---+-----+
only showing top 20 rows



In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("JoinStagesExample").getOrCreate()

# Create two DataFrames with 4 partitions each
df1 = spark.range(1, 10001).repartition(4)
df2 = spark.range(5000, 15001).repartition(4)

# Stage 1: Narrow transformation (repartition)
df1_filtered = df1.filter(col("id") % 2 == 0)
df2_filtered = df2.filter(col("id") % 3 == 0)

# Stage 2: Wide transformation - join (causes shuffle)
joined_df = df1_filtered.join(df2_filtered, on="id", how="inner")

# Stage 3: Narrow transformation - select columns
final_df = joined_df.select("id")

# Action to trigger execution
result = final_df.show()




+----+
|  id|
+----+
|5376|
|7620|
|7590|
|9000|
|5358|
|5250|
|8382|
|5112|
|5388|
|6768|
|6690|
|5832|
|9612|
|8754|
|9852|
|8418|
|8322|
|9996|
|6312|
|8898|
+----+
only showing top 20 rows



In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("RDDJoinStagesExample").getOrCreate()
sc = spark.sparkContext

# Create two RDDs with 4 partitions each
rdd1 = sc.parallelize(range(1, 10001), 4).map(lambda x: (x, x))
rdd2 = sc.parallelize(range(5000, 15001), 4).map(lambda x: (x, x))

# Stage 1: Narrow transformations (map)
rdd1_filtered = rdd1.filter(lambda x: x[0] % 2 == 0)
rdd2_filtered = rdd2.filter(lambda x: x[0] % 3 == 0)

# Stage 2: Wide transformation - join (causes shuffle)
joined_rdd = rdd1_filtered.join(rdd2_filtered)  # join on key

# Stage 3: Narrow transformation (map)
final_rdd = joined_rdd.map(lambda x: x[0])  # select the key

# Action to trigger execution
result = final_rdd.collect()



In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("RDDBasedCaching").getOrCreate()
sc = spark.sparkContext

# Create RDD
rdd = sc.parallelize(range(1, 10001), 4)

# Perform some transformation
squared_rdd = rdd.map(lambda x: (x, x * x))

# Cache the RDD
squared_rdd.cache()

# Trigger an action (1st time: cache will be populated)



PythonRDD[21] at RDD at PythonRDD.scala:53

In [ ]:
print(squared_rdd.count())

# 2nd action (uses cached data)
print(squared_rdd.take(5))

10000
[(1, 1), (2, 4), (3, 9), (4, 16), (5, 25)]


In [ ]:
from pyspark.sql.functions import col

# Create DataFrame
df = spark.range(1, 10001).withColumn("square", col("id") * col("id"))

# Cache the DataFrame
df.cache()

# Trigger action to materialize the cache
df.count()

# Next action will use cached result
df.show(5)

+---+------+
| id|square|
+---+------+
|  1|     1|
|  2|     4|
|  3|     9|
|  4|    16|
|  5|    25|
+---+------+
only showing top 5 rows



In [ ]:
df.unpersist()
squared_rdd.unpersist()

PythonRDD[21] at RDD at PythonRDD.scala:53